# COINS 03 - Final LR/SVM Actual Run Results

This notebook starts with the actual run-level results, not coverage. Each COINS result row is one dataset × seed × noise protocol run. Expected entries: 15 datasets × 10 seeds × 3 protocols = 450 rows per model.


In [1]:
from __future__ import annotations
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm, wilcoxon
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 260)
ROOT = Path.cwd()
RAW_RESULTS = ROOT / "docs" / "experiments" / "raw" / "full-benchmark-solution-v2.csv"
results = pd.read_csv(RAW_RESULTS)
lr_svm = results[results["model"].isin(["lr", "svm"])].copy()
print("source", RAW_RESULTS)
print("full rows", len(results))
print("LR/SVM rows", len(lr_svm))


source /home/than-minh/project/DSP/docs/experiments/raw/full-benchmark-solution-v2.csv
full rows 24750
LR/SVM rows 6300


## Actual COINS Run Results: 450 Entries Per Model

These are the actual final COINS training results (`cwms_msbs`) for LR and SVM. They include BA, macro F1, minority/majority precision, and minority/majority recall for every dataset/seed/noise-protocol run.


In [2]:
coins_runs = lr_svm[lr_svm["method"] == "cwms_msbs"].copy()
run_cols = [
    "model", "dataset", "seed", "noise_protocol", "method",
    "balanced_accuracy", "macro_f1",
    "minority_precision", "majority_precision",
    "minority_recall", "majority_recall",
]
# Some older result files may not carry majority_precision. Compute a placeholder only if absent.
if "majority_precision" not in coins_runs.columns:
    coins_runs["majority_precision"] = np.nan
coins_run_results = coins_runs[run_cols].sort_values(["model", "dataset", "seed", "noise_protocol"]).reset_index(drop=True)
display(coins_run_results)

run_count = coins_run_results.groupby("model").size().reset_index(name="actual_entries")
run_count["expected_entries"] = 15 * 10 * 3
display(run_count)


,model,dataset,seed,noise_protocol,method,balanced_accuracy,macro_f1,minority_precision,majority_precision,minority_recall,majority_recall
0,lr,abalone,13,hidden_minority_high,cwms_msbs,0.694101,0.670706,0.526749,NaN,0.739884,0.648318
1,lr,abalone,13,hidden_minority_low,cwms_msbs,0.712247,0.709575,0.609890,NaN,0.641618,0.782875
2,lr,abalone,13,hidden_minority_medium,cwms_msbs,0.725504,0.722677,0.626374,NaN,0.658960,0.792049
3,lr,abalone,17,hidden_minority_high,cwms_msbs,0.782486,0.753030,0.609053,NaN,0.855491,0.709480
4,lr,abalone,17,hidden_minority_low,cwms_msbs,0.764252,0.763892,0.689655,NaN,0.693642,0.834862
...,...,...,...,...,...,...,...,...,...,...,...
895,svm,yeast,47,hidden_minority_low,cwms_msbs,0.723189,0.761047,0.763158,NaN,0.475410,0.970968
896,svm,yeast,47,hidden_minority_medium,cwms_msbs,0.693628,0.735689,0.781250,NaN,0.409836,0.977419
897,svm,yeast,53,hidden_minority_high,cwms_msbs,0.606822,0.631736,0.600000,NaN,0.245902,0.967742
898,svm,yeast,53,hidden_minority_low,cwms_msbs,0.760947,0.790881,0.755556,NaN,0.557377,0.964516


,model,actual_entries,expected_entries
0,lr,450,450
1,svm,450,450


## Actual No-Cleaning Baseline Run Results: 450 Entries Per Model

Same run-level layout for the raw no-cleaning baseline, so COINS has a direct row-level reference.


In [3]:
base_runs = lr_svm[lr_svm["method"] == "no_cleaning"].copy()
if "majority_precision" not in base_runs.columns:
    base_runs["majority_precision"] = np.nan
base_run_results = base_runs[run_cols].sort_values(["model", "dataset", "seed", "noise_protocol"]).reset_index(drop=True)
display(base_run_results)


,model,dataset,seed,noise_protocol,method,balanced_accuracy,macro_f1,minority_precision,majority_precision,minority_recall,majority_recall
0,lr,abalone,13,hidden_minority_high,no_cleaning,0.527373,0.453858,0.909091,NaN,0.057803,0.996942
1,lr,abalone,13,hidden_minority_low,no_cleaning,0.593511,0.573468,0.875000,NaN,0.202312,0.984709
2,lr,abalone,13,hidden_minority_medium,no_cleaning,0.550326,0.499920,0.863636,NaN,0.109827,0.990826
3,lr,abalone,17,hidden_minority_high,no_cleaning,0.511561,0.419926,1.000000,NaN,0.023121,1.000000
4,lr,abalone,17,hidden_minority_low,no_cleaning,0.565970,0.530603,0.812500,NaN,0.150289,0.981651
...,...,...,...,...,...,...,...,...,...,...,...
895,svm,yeast,47,hidden_minority_low,no_cleaning,0.642702,0.684664,0.857143,NaN,0.295082,0.990323
896,svm,yeast,47,hidden_minority_medium,no_cleaning,0.552538,0.556118,0.700000,NaN,0.114754,0.990323
897,svm,yeast,53,hidden_minority_high,no_cleaning,0.496774,0.453608,0.000000,NaN,0.000000,0.993548
898,svm,yeast,53,hidden_minority_low,no_cleaning,0.682073,0.729254,0.851852,NaN,0.377049,0.987097


## Run-Level COINS vs No-Cleaning Delta

Paired by model, dataset, seed, and protocol. Positive delta means COINS improved that metric for the same run.


In [4]:
key = ["model", "dataset", "seed", "noise_protocol"]
paired = coins_run_results.set_index(key).join(base_run_results.set_index(key), lsuffix="_coins", rsuffix="_no_cleaning", how="inner").reset_index()
for metric in ["balanced_accuracy", "macro_f1", "minority_recall", "majority_recall"]:
    paired[f"delta_{metric}"] = paired[f"{metric}_coins"] - paired[f"{metric}_no_cleaning"]
display(paired[[*key, "delta_balanced_accuracy", "delta_macro_f1", "delta_minority_recall", "delta_majority_recall"]])


,model,dataset,seed,noise_protocol,delta_balanced_accuracy,delta_macro_f1,delta_minority_recall,delta_majority_recall
0,lr,abalone,13,hidden_minority_high,0.166729,0.216848,0.682081,-0.348624
1,lr,abalone,13,hidden_minority_low,0.118736,0.136107,0.439306,-0.201835
2,lr,abalone,13,hidden_minority_medium,0.175178,0.222757,0.549133,-0.198777
3,lr,abalone,17,hidden_minority_high,0.270925,0.333104,0.832370,-0.290520
4,lr,abalone,17,hidden_minority_low,0.198282,0.233289,0.543353,-0.146789
...,...,...,...,...,...,...,...,...
895,svm,yeast,47,hidden_minority_low,0.080487,0.076384,0.180328,-0.019355
896,svm,yeast,47,hidden_minority_medium,0.141089,0.179572,0.295082,-0.012903
897,svm,yeast,53,hidden_minority_high,0.110048,0.178127,0.245902,-0.025806
898,svm,yeast,53,hidden_minority_low,0.078874,0.061627,0.180328,-0.022581


## Aggregated Dataset Table: COINS vs No Cleaning

Mean over 10 seeds × 3 protocols for each dataset and model.


In [5]:
metric_cols = ["balanced_accuracy", "macro_f1", "minority_precision", "minority_recall", "majority_recall"]
agg_dataset = lr_svm[lr_svm["method"].isin(["cwms_msbs", "no_cleaning"])].groupby(["model", "dataset", "method"])[metric_cols].mean().reset_index()
display(agg_dataset.sort_values(["model", "dataset", "method"]))


,model,dataset,method,balanced_accuracy,macro_f1,minority_precision,minority_recall,majority_recall
0,lr,abalone,cwms_msbs,0.738683,0.725750,0.610951,0.728131,0.749235
1,lr,abalone,no_cleaning,0.556175,0.507369,0.892595,0.122543,0.989806
2,lr,blood,cwms_msbs,0.662514,0.632165,0.422386,0.598956,0.726071
3,lr,blood,no_cleaning,0.517495,0.468632,0.770000,0.036633,0.998357
4,lr,breast_cancer,cwms_msbs,0.871873,0.860222,0.785630,0.895597,0.848148
5,lr,breast_cancer,no_cleaning,0.799993,0.814921,0.966512,0.612579,0.987407
6,lr,credit-g,cwms_msbs,0.637746,0.620243,0.448364,0.600444,0.675048
7,lr,credit-g,no_cleaning,0.542127,0.501895,0.684853,0.104444,0.979810
8,lr,ecoli,cwms_msbs,0.850121,0.777549,0.560388,0.922807,0.777436
9,lr,ecoli,no_cleaning,0.716694,0.746932,0.948117,0.442105,0.991282


## Comparative Training: LR/SVM COINS vs Other Methods

This compares COINS with every LR/SVM method in the cached full benchmark.


In [6]:
all_metric_cols = ["balanced_accuracy", "macro_f1", "minority_precision", "minority_recall", "majority_recall", "pr_auc"]
display(lr_svm.groupby(["model", "method"])[all_metric_cols].mean().reset_index().sort_values(["model", "balanced_accuracy"], ascending=[True, False]))
display(lr_svm.groupby(["model", "method", "noise_protocol"])[all_metric_cols].mean().reset_index().sort_values(["model", "noise_protocol", "balanced_accuracy"], ascending=[True, True, False]))
display(lr_svm.pivot_table(index=["model", "dataset"], columns="method", values="balanced_accuracy", aggfunc="mean").reset_index().sort_values(["model", "dataset"]))


,model,method,balanced_accuracy,macro_f1,minority_precision,minority_recall,majority_recall,pr_auc
2,lr,cwms_msbs,0.734133,0.704454,0.545171,0.716009,0.752258,0.637694
3,lr,cwms_msbs_shuffled,0.716811,0.704790,0.597786,0.604218,0.829404,0.640702
1,lr,cwms,0.715478,0.711278,0.623656,0.557669,0.873287,0.642490
0,lr,class_proportional,0.702488,0.703100,0.654241,0.489710,0.915265,0.649156
4,lr,msbs,0.648319,0.641836,0.693285,0.335896,0.960741,0.638457
6,lr,smote,0.643809,0.634726,0.704718,0.321399,0.966220,0.640964
5,lr,no_cleaning,0.599602,0.572740,0.682305,0.210305,0.988899,0.644871
9,svm,cwms_msbs,0.676570,0.670124,0.691339,0.398869,0.954270,0.660248
7,svm,class_proportional,0.672893,0.660773,0.664122,0.374155,0.971631,0.641813
10,svm,cwms_msbs_shuffled,0.668310,0.660318,0.691728,0.373638,0.962982,0.656295


,model,method,noise_protocol,balanced_accuracy,macro_f1,minority_precision,minority_recall,majority_recall,pr_auc
9,lr,cwms_msbs_shuffled,hidden_minority_high,0.703927,0.668233,0.500339,0.698259,0.709596,0.598208
6,lr,cwms_msbs,hidden_minority_high,0.703146,0.655579,0.470080,0.750133,0.656159,0.598881
3,lr,cwms,hidden_minority_high,0.699229,0.688117,0.563773,0.572244,0.826213,0.606638
0,lr,class_proportional,hidden_minority_high,0.681061,0.680242,0.613631,0.455737,0.906384,0.613283
12,lr,msbs,hidden_minority_high,0.616994,0.604727,0.664936,0.275644,0.958344,0.593035
18,lr,smote,hidden_minority_high,0.610930,0.593859,0.667337,0.256819,0.965042,0.598195
15,lr,no_cleaning,hidden_minority_high,0.567038,0.529361,0.600617,0.145248,0.988827,0.604251
7,lr,cwms_msbs,hidden_minority_low,0.759420,0.739592,0.597973,0.710791,0.808049,0.671736
4,lr,cwms,hidden_minority_low,0.737252,0.736970,0.663842,0.580622,0.893883,0.675243
10,lr,cwms_msbs_shuffled,hidden_minority_low,0.733981,0.734969,0.666427,0.571493,0.896469,0.676797


method,model,dataset,class_proportional,cwms,cwms_msbs,cwms_msbs_shuffled,msbs,no_cleaning,smote
0,lr,abalone,0.685868,0.709224,0.738683,0.717644,0.611102,0.556175,0.600832
1,lr,blood,0.583298,0.585354,0.662514,0.615551,0.537425,0.517495,0.533457
2,lr,breast_cancer,0.891978,0.892467,0.871873,0.875405,0.842082,0.799993,0.839605
3,lr,credit-g,0.611841,0.631016,0.637746,0.624476,0.580508,0.542127,0.583746
4,lr,ecoli,0.843360,0.846370,0.850121,0.847665,0.801997,0.716694,0.795533
5,lr,glass_float,0.564815,0.613426,0.662963,0.615278,0.544444,0.513889,0.544444
6,lr,haberman,0.588596,0.577281,0.597164,0.572427,0.516257,0.505570,0.515307
7,lr,ilpd,0.526786,0.541819,0.594368,0.563744,0.506647,0.498634,0.503617
8,lr,ionosphere,0.738914,0.752530,0.732143,0.729836,0.729836,0.703720,0.723363
9,lr,kc1,0.654211,0.675630,0.700014,0.687186,0.592242,0.551242,0.594199
